# Graph-Grounded Reddit Emotion Intelligence RAG

A reproducible portfolio notebook for customer-interaction-style analytics: classify emotional polarity, build conversation graphs, retrieve evidence with LangChain, and produce grounded summaries.

**Important:** the reported benchmark is a binary polarity aggregation of GoEmotions labels. It is not the original 28-class benchmark.

## Business problem

Support and complaint teams need more than a sentiment score: they need to find representative evidence, understand which communities and conversation branches disagree, and inspect changes over time. This notebook demonstrates a graph-grounded RAG workflow for that use case.

In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

DATA = Path('../data/goemotions')
OUT = Path('../outputs/notebook_showcase')
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.concat([pd.read_csv(p) for p in glob.glob(str(DATA / 'goemotions_*.csv'))], ignore_index=True)
raw.shape

In [ ]:
# Collapse rater rows to one high-confidence polarity label per comment.
emotion_cols = [c for c in raw.columns if c not in {'text','id','author','subreddit','link_id','parent_id','created_utc','rater_id','example_very_unclear'}]
positive = set('admiration amusement approval caring desire excitement gratitude joy love optimism pride relief'.split())
negative = set('anger annoyance disappointment disapproval disgust embarrassment fear grief nervousness remorse sadness'.split())
votes = raw.groupby('id')[emotion_cols].sum()
texts = raw.drop_duplicates('id').set_index('id')
rows = []
for comment_id, row in votes.iterrows():
    labels = [e for e, value in row.items() if value == row.max() and value > 0]
    if len(labels) == 1 and labels[0] in positive | negative:
        record = texts.loc[comment_id]
        rows.append({'text': record.text, 'label': int(labels[0] in positive), 'id': comment_id, 'author': record.author, 'subreddit': record.subreddit, 'parent_id': record.parent_id, 'created_utc': record.created_utc})
data = pd.DataFrame(rows)
data.label.value_counts().sort_index()

In [ ]:
# Leakage-safe stratified benchmark. The test set is never used during fitting.
train, test = train_test_split(data, test_size=.2, random_state=42, stratify=data.label)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=30000)
X_train = vectorizer.fit_transform(train.text)
X_test = vectorizer.transform(test.text)
model = LinearSVC(C=1)
model.fit(X_train, train.label)
pred = model.predict(X_test)
metrics = {'accuracy': accuracy_score(test.label, pred), 'macro_f1': f1_score(test.label, pred, average='macro'), 'weighted_f1': f1_score(test.label, pred, average='weighted')}
metrics

In [ ]:
print(classification_report(test.label, pred, target_names=['negative','positive']))
ConfusionMatrixDisplay.from_predictions(test.label, pred, display_labels=['negative','positive'], cmap='Blues')
plt.title('Held-out polarity confusion matrix')
plt.tight_layout()
plt.savefig(OUT / 'confusion_matrix.png', dpi=160)
plt.show()

In [ ]:
# Conversation graph: parent-child edges and subreddit/community summaries.
graph = nx.DiGraph()
for row in data.dropna(subset=['parent_id']).itertuples():
    graph.add_edge(str(row.parent_id), str(row.id), subreddit=row.subreddit, label=row.label)
print({'nodes': graph.number_of_nodes(), 'edges': graph.number_of_edges(), 'weak_components': nx.number_weakly_connected_components(graph)})
community = data.groupby('subreddit').agg(comments=('id','count'), positive_rate=('label','mean')).sort_values('comments', ascending=False).head(15)
community.plot.bar(y='positive_rate', legend=False, figsize=(10,4), title='Positive polarity rate by largest communities')
plt.ylabel('positive rate'); plt.tight_layout(); plt.savefig(OUT / 'community_polarity.png', dpi=160); plt.show()

## LangChain RAG

The production code in `src/reddit_rag/rag.py` uses a LangChain `BaseRetriever`, BM25 + SPLADE reciprocal-rank fusion, Qdrant sparse vectors, and evidence-cited extractive answers. Run it from the repository root with:

```powershell
$env:PYTHONPATH = (Join-Path (Get-Location) 'src')
.\.venv_sys\Scripts\python.exe -m reddit_rag.rag_cli --politosphere-comments .\data\politosphere\comments_2008-01.bz2 --query "What concerns appear in the discussion?" --limit 100 --top-k 5
```

The answer is intentionally grounded in retrieved evidence and should be reviewed by an analyst; it is not a population-level opinion estimate.